# Computation of Energy Distribution Function

## Imports

In [ ]:
import ROOT, glob
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import pandas as pd

plt.rcParams['figure.dpi'] = 400
plt.rcParams['savefig.dpi'] = 400

## Load data

In [2]:
dirDir= "/home/users/l/lebrun/"
dataDir = "/dpnc/beegfs/dampe/tryf/MC_Skimmed/MC_gamma/"
resultPath = dirDir + "results/"
dataPath = dirDir + "data/"
filename = dataDir + "DmlNtup_gamma_part000_TQ_FullSel.root"

# Find all matching ROOT files : from 1 GeV to 100 GeV
files = sorted(glob.glob(dataDir + "DmlNtup_gamma_part*_TQ_FullSel.root"))

# Pass the list of files to RDataFrame
rdf = ROOT.RDataFrame("DmlNtup", files)

# Extract only desired columns
df = rdf.AsNumpy(["tt_ekin", "tt_bgoTotalEcorr_GeV", "tt_theta", "tt_phi", "theta_True", "phi_True", "theta_BestT", "phi_BestT", "tt_isHET", "tt_isLET"])
df = pd.DataFrame(df)

In [3]:
df['Etrue'] = df['tt_ekin'] * 1e-3
df['Erec'] = df['tt_bgoTotalEcorr_GeV']

df['theta_True_deg']= df['theta_True']
df['phi_True_deg']= df['phi_True']
df['theta_True'] = np.deg2rad(df['theta_True'])
df['phi_True'] = np.deg2rad(df['phi_True'])

## Compute energy deviation

In [4]:
df['Ediff'] = df['Etrue'] - df['Erec']
df['theta_True_deg'] = np.rad2deg(df['theta_True'])

## Binning (obsolete?)

In [5]:
from dataclasses import dataclass

@dataclass
class Bin:
    edges: tuple
    center: float
    bin_width: float
    value: any = None  # can hold a histogram or anything

    def __repr__(self):
        return f"Bin(edges={self.edges}, center={self.center}, bin_width={self.bin_width}, value={self.value})"

    def min(self):
        return np.min(self.edges)

    def max(self):
        return np.max(self.edges)

class Bins1D:
    def __init__(self, edges, values=None):
        if not isinstance(edges, np.ndarray):
            edges = np.array(edges)
        self.edges = edges
        self.bins = []
        for i in range(len(edges) - 1):
            center = (edges[i] + edges[i+1]) / 2
            width = edges[i+1] - edges[i]
            value = values[i] if values is not None else None
            self.bins.append(Bin((edges[i], edges[i+1]), center, width, value))

    def __getitem__(self, idx):
        return self.bins[idx]
    
    def __len__(self):
        return len(self.bins)

    def centers(self):
        return np.array([bin.center for bin in self.bins])

    def bin_edges(self):
        return self.edges

    def get_bin_index(self, value):
        idx = np.searchsorted(self.edges, value, side='right') - 1
        if 0 <= idx < len(self.bins):
            return idx
        return None

    def __repr__(self):
        return f"Bins1D({self.bins})"

class Bins2D:
    def __init__(self, edges_x, edges_y):
        self.x_bins = Bins1D(edges_x)
        self.y_bins = Bins1D(edges_y)
        
        # Create 2D matrix of empty values
        self.grid = np.empty((len(self.x_bins), len(self.y_bins)), dtype=object)
        for i in range(len(self.x_bins)):
            for j in range(len(self.y_bins)):
                self.grid[i, j] = None  # You can initialize with empty histogram if needed

    def __getitem__(self, idx):
        if isinstance(idx, tuple) and len(idx) == 2:
            i, j = idx
            return self.grid[i, j]
        elif isinstance(idx, int):
            return self.grid[idx, :]
        elif isinstance(idx, slice):
            return self.grid[idx, :]
        else:
            raise IndexError("Invalid index type")
    
    def __setitem__(self, idx, value):
        if isinstance(idx, tuple) and len(idx) == 2:
            i, j = idx
            self.grid[i, j] = value
        else:
            raise IndexError("Must use two indices to set a value")
    
    def shape(self):
        return self.grid.shape

    def populate_from_dataframe(self, df, E_col, theta_col, value_col, hist_func):
        for _, row in df.iterrows():
            i = self.x_bins.get_bin_index(row[E_col])
            j = self.y_bins.get_bin_index(row[theta_col])
            if i is not None and j is not None:
                if self.grid[i, j] is None:
                    self.grid[i, j] = []
                self.grid[i, j].append(row[value_col])
        
        for i in range(self.shape()[0]):
            for j in range(self.shape()[1]):
                if self.grid[i, j] is not None:
                    self.grid[i, j] = hist_func(self.grid[i, j])
                else:
                    self.grid[i, j] = hist_func([])

    def __repr__(self):
        return f"Bins2D(shape={self.shape()})"


## Separation of HET and LET triggers

In [ ]:
# Define the two datasets based on the tt_isHET flag
dfHET = df[df["tt_isHET"] == 1].copy()
dfLET = df[df["tt_isLET"] == 1].copy()

scratchDir = dirDir + "scratch/"
#dfHET.to_csv(scratchDir + "dfHET.csv", index=False)
#dfLET.to_csv(scratchDir + "dfLET.csv", index=False)

In [9]:

print(f"Number of HET events: {len(dfHET)}")
print(f"Number of LET events: {len(dfLET)}")

print(f"Max LET energy: {dfLET['Etrue'].max()}")
print(f"Max HET energy: {dfHET['Etrue'].max()}")
print(f"Min LET energy: {dfLET['Etrue'].min()}")
print(f"Min HET energy: {dfHET['Etrue'].min()}")

Number of HET events: 7252603
Number of LET events: 13021387
Max LET energy: 99.99998402010318
Max HET energy: 99.99998402010318
Min LET energy: 1.000019642021374
Min HET energy: 1.220147597224916


In [8]:

# Energy bins for LET
Emin = df['Etrue'].min()
Emax = df['Etrue'].max()
n_E_bins_LET = 4
n_E_bins_HET = 8

# Define theta bins for both LET and HET
theta_min = 0
theta_max = 60
n_theta_bins = 5
theta_bins_edge = np.linspace(theta_min, theta_max, n_theta_bins + 1)

# Set up for processing both datasets
datasets = [("LET", dfLET, n_E_bins_LET), ("HET", dfHET, n_E_bins_HET)]

for dataset_name, dataset, n_E_bins in datasets:
    print(f"Processing {dataset_name} dataset...")

    # Define energy bin edges for the dataset
    E_bins_edge = np.logspace(np.log10(Emin), np.log10(Emax), n_E_bins + 1)
    E_bins_center = np.sqrt(E_bins_edge[:-1] * E_bins_edge[1:])
    
    print(f"E_bins for {dataset_name}:", E_bins_center)
    print(f"E_bins_edge for {dataset_name}:", E_bins_edge)

    # Assign bin indices for energy and theta bins
    dataset['E_bin'] = pd.cut(dataset['Etrue'], bins=E_bins_edge, labels=False, include_lowest=True)
    dataset['theta_bin'] = pd.cut(dataset['theta_True_deg'], bins=theta_bins_edge, labels=False, include_lowest=True)

    # Drop rows that fall outside the bins (e.g., NaN bin index)
    dataset = dataset.dropna(subset=['E_bin', 'theta_bin'])

    # Make sure bin indices are integers
    dataset['E_bin'] = dataset['E_bin'].astype(int)
    dataset['theta_bin'] = dataset['theta_bin'].astype(int)

    # Group by (E_bin, theta_bin)
    grouped = dataset.groupby(['E_bin', 'theta_bin'])

    # Apply resolution calculation: HBW of 68% containment interval 
    resolution_per_bin = grouped['Ediff'].apply(
        lambda x: 0.5 * (np.percentile(x, 84) - np.percentile(x, 16))
    ).reset_index(name='resolution')

    # Merge back the resolution into the original dataframe
    dataset = dataset.merge(resolution_per_bin, on=['E_bin', 'theta_bin'], how='left')

    # Print out a few rows to verify the results
    print(dataset.head())



Processing LET dataset...
E_bins for LET: [ 1.77828035  5.62341513 17.78279656 56.23412928]
E_bins_edge for LET: [ 1.00000063  3.16227903 10.00000236 31.62277788 99.99998444]


/tmp/ipykernel_465202/1324011860.py:34: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dataset['E_bin'] = dataset['E_bin'].astype(int)
/tmp/ipykernel_465202/1324011860.py:35: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dataset['theta_bin'] = dataset['theta_bin'].astype(int)


        tt_ekin  tt_bgoTotalEcorr_GeV  tt_theta    tt_phi     theta_True  \
0  11948.103856             12.116264  0.749892  0.072068  1.932423e-312   
1  70266.252720             69.955447  0.936644  0.515106  1.932423e-312   
2   6449.162407              6.249581  1.065695  2.840578  1.932423e-312   
3  51479.672419             51.234132  0.241034 -2.105158   2.410338e-01   
4  12568.459228             12.276070  0.342936  1.639515   3.429358e-01   

        phi_True    theta_BestT      phi_BestT  tt_isHET  tt_isLET      Etrue  \
0  1.649542e-317  1.107197e-310  8.819582e-316       0.0       1.0  11.948104   
1  1.649542e-317  1.107197e-310  8.819582e-316       1.0       1.0  70.266253   
2  1.649542e-317  1.107197e-310  8.819582e-316       1.0       1.0   6.449162   
3  -2.105158e+00  1.107197e-310  8.819582e-316       0.0       1.0  51.479672   
4   1.639515e+00  1.107197e-310  8.819582e-316       1.0       1.0  12.568459   

        Erec  theta_True_deg   phi_True_deg     Ediff  E

/tmp/ipykernel_465202/1324011860.py:34: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dataset['E_bin'] = dataset['E_bin'].astype(int)
/tmp/ipykernel_465202/1324011860.py:35: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dataset['theta_bin'] = dataset['theta_bin'].astype(int)


        tt_ekin  tt_bgoTotalEcorr_GeV  tt_theta    tt_phi     theta_True  \
0  70266.252720             69.955447  0.936644  0.515106  1.932423e-312   
1   6449.162407              6.249581  1.065695  2.840578  1.932423e-312   
2  12568.459228             12.276070  0.342936  1.639515   3.429358e-01   
3  17004.097695             16.558367  1.244274  1.409811   3.429358e-01   
4  60288.367372             53.226870  0.842224 -3.100587   8.422239e-01   

        phi_True    theta_BestT      phi_BestT  tt_isHET  tt_isLET      Etrue  \
0  1.649542e-317  1.107197e-310  8.819582e-316       1.0       1.0  70.266253   
1  1.649542e-317  1.107197e-310  8.819582e-316       1.0       1.0   6.449162   
2   1.639515e+00  1.107197e-310  8.819582e-316       1.0       1.0  12.568459   
3   1.639515e+00  1.107197e-310  8.819582e-316       1.0       1.0  17.004098   
4  -3.100587e+00  1.107197e-310  8.819582e-316       1.0       1.0  60.288367   

        Erec  theta_True_deg   phi_True_deg     Ediff  E